In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

preprocessor = joblib.load("../models/preprocessor.pkl")

In [2]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [3]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [4]:
dummy = DummyRegressor(strategy='mean')

dummy_scores = cross_val_score(dummy, X_train_processed, y_train,
                                 cv=cv, scoring='neg_root_mean_squared_error')

print("DummyRegressor CV RMSE (log-scale):", -dummy_scores.mean())
print("Std:", dummy_scores.std())

DummyRegressor CV RMSE (log-scale): 0.3973154735420289
Std: 0.009273887013736444


In [5]:
linreg = LinearRegression()

linreg_scores = cross_val_score(linreg, X_train_processed, y_train,
                                  cv=cv, scoring='neg_root_mean_squared_error')

print("Linear Regression CV RMSE (log-scale):", -linreg_scores.mean())
print("Std:", linreg_scores.std())

Linear Regression CV RMSE (log-scale): 0.12964162594924086
Std: 0.010090402068916891


In [6]:
ridge = Ridge(alpha=1.0, random_state=42)

ridge_scores = cross_val_score(ridge, X_train_processed, y_train,
                                 cv=cv, scoring='neg_root_mean_squared_error')

print("Ridge CV RMSE (log-scale):", -ridge_scores.mean())
print("Std:", ridge_scores.std())

Ridge CV RMSE (log-scale): 0.11955439347185619
Std: 0.00840645830536626


In [7]:
results = pd.DataFrame({
    'Model': ['DummyRegressor', 'Linear Regression', 'Ridge'],
    'CV_RMSE_mean': [-dummy_scores.mean(), -linreg_scores.mean(), -ridge_scores.mean()],
    'CV_RMSE_std': [dummy_scores.std(), linreg_scores.std(), ridge_scores.std()]
})
print(results)

               Model  CV_RMSE_mean  CV_RMSE_std
0     DummyRegressor      0.397315     0.009274
1  Linear Regression      0.129642     0.010090
2              Ridge      0.119554     0.008406
